# Notebook 1 — Anomaly Detection Model (FSR + Lid-State Fusion)
**IoT-Based Smart Waste Management System — Urban Nigeria**

---

## Overview
This notebook implements and evaluates the dual-sensor anomaly detection subsystem described in Section 4.2 of the paper.

The system detects **unauthorized dumping** by fusing:
- **Force-Sensitive Resistor (FSR)** readings — detects sudden weight increases
- **Lid-state flag** — distinguishes authorized (lid open) vs. unauthorized (lid closed) loading
- **Temporal gate** — excludes brief actuator delays

### Anomaly Indicator Function (Eq. 5)
$$A_e(t) = \mathbb{1}\left[(\Delta W(t) > \theta_f) \wedge (L(t) = 0) \wedge (\Delta t_{lid} > 5s)\right]$$

### Calibrated Threshold (Eq. 4)
$$\theta_f = \mu + 2\sigma = 0.04 + 2(0.21) \approx 0.5 \text{ kg}$$


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (roc_curve, roc_auc_score,
                             confusion_matrix, classification_report)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")


## 1. Baseline Force Noise Characterisation
The FSR noise distribution was established from a 24-hour idle baseline log
sampled at 10 Hz, yielding N(μ = 0.04, σ = 0.21) kg.
The anomaly threshold is set at μ + 2σ ≈ 0.46 kg → rounded to **0.5 kg**.


In [ ]:
# ── Baseline noise parameters (from 24-hour empirical log) ───────────────────
MU_NOISE   = 0.04   # kg  — mean idle FSR reading
SIGMA_NOISE = 0.21  # kg  — std dev of idle FSR reading
THETA_F    = round(MU_NOISE + 2 * SIGMA_NOISE, 2)   # calibrated threshold
THETA_F    = 0.5    # rounded operational value

print(f"Noise distribution : N(μ={MU_NOISE}, σ={SIGMA_NOISE}) kg")
print(f"Theoretical threshold (μ + 2σ) : {MU_NOISE + 2*SIGMA_NOISE:.3f} kg")
print(f"Operational threshold (θ_f)    : {THETA_F} kg")
print(f"Theoretical FPR under Gaussian : {(1 - 0.9772)*100:.2f}%")


## 2. Simulate Disposal Events
We simulate 165 disposal events split into:
- **45 unauthorized dumps** — large sudden weight spike, lid closed
- **120 authorized disposals** — gradual loading, lid open

Each event yields a `ΔW` (force delta) and a lid-state flag `L`.


In [ ]:
# ── Event simulation ─────────────────────────────────────────────────────────
N_UNAUTHORIZED = 45
N_AUTHORIZED   = 120
LID_DELAY_GATE = 5.0   # seconds — minimum lid-open duration to be 'authorized'

# Unauthorized dumps: large ΔW spike, lid stays closed
dW_unauth = np.random.normal(loc=1.8, scale=0.4, size=N_UNAUTHORIZED)
dW_unauth = np.clip(dW_unauth, 0.3, 4.0)
lid_unauth = np.zeros(N_UNAUTHORIZED)           # lid closed = 0
dt_unauth  = np.random.uniform(0, 3, N_UNAUTHORIZED)  # short delay (< gate)

# Authorized disposals: moderate ΔW, lid open
dW_auth = np.random.normal(loc=0.6, scale=0.3, size=N_AUTHORIZED)
dW_auth = np.clip(dW_auth, 0.0, 1.5)
lid_auth = np.ones(N_AUTHORIZED)               # lid open = 1
dt_auth  = np.random.uniform(5, 30, N_AUTHORIZED)  # lid held open > gate

# Combine
dW_all  = np.concatenate([dW_unauth,  dW_auth])
lid_all = np.concatenate([lid_unauth, lid_auth])
dt_all  = np.concatenate([dt_unauth,  dt_auth])
y_true  = np.concatenate([np.ones(N_UNAUTHORIZED), np.zeros(N_AUTHORIZED)])

print(f"Total events : {len(y_true)}")
print(f"  Unauthorized (positive) : {N_UNAUTHORIZED}")
print(f"  Authorized   (negative) : {N_AUTHORIZED}")


## 3. Apply Anomaly Indicator Function (Eq. 5)
$$A_e(t) = \mathbb{1}\left[(\Delta W > \theta_f) \;\wedge\; (L = 0) \;\wedge\; (\Delta t_{lid} > 5s)\right]$$


In [ ]:
# ── Anomaly detection logic ───────────────────────────────────────────────────
def detect_anomaly(dW, lid_state, dt_lid, theta_f=THETA_F, gate=LID_DELAY_GATE):
    """
    Returns 1 (anomaly) if all three conditions are met:
      1. Force delta exceeds calibrated threshold
      2. Lid is closed (lid_state = 0)
      3. No recent authorized lid-open event (dt_lid > gate)
    """
    cond_force = dW > theta_f
    cond_lid   = lid_state == 0
    cond_time  = dt_lid > gate
    return (cond_force & cond_lid & cond_time).astype(int)

y_pred = detect_anomaly(dW_all, lid_all, dt_all)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
precision  = tp / (tp + fp)
recall     = tp / (tp + fn)
f1         = 2 * precision * recall / (precision + recall)
specificity = tn / (tn + fp)

print("=" * 45)
print(f"  TP = {tp:3d}   FN = {fn:3d}")
print(f"  FP = {fp:3d}   TN = {tn:3d}")
print("=" * 45)
print(f"  Precision   : {precision*100:.1f}%")
print(f"  Recall      : {recall*100:.1f}%")
print(f"  F1-score    : {f1*100:.1f}%")
print(f"  Specificity : {specificity*100:.1f}%")
print(f"  Observed FPR: {fp/(fp+tn)*100:.1f}%")


## 4. ROC Curve and AUC
Sweep θ_f across [0.1, 2.0] kg to construct the full ROC curve.

In [ ]:
# ── ROC curve by sweeping threshold ──────────────────────────────────────────
# Compute a continuous 'anomaly score' = ΔW * (1 - lid_state)
anomaly_score = dW_all * (1 - lid_all)

fpr_arr, tpr_arr, thresholds = roc_curve(y_true, anomaly_score)
auc_val = roc_auc_score(y_true, anomaly_score)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC
axes[0].plot(fpr_arr, tpr_arr, color='#1A5276', lw=2.5,
             label=f'ROC (AUC = {auc_val:.3f})')
axes[0].plot([0,1],[0,1],'--', color='#AAAAAA', lw=1)
axes[0].fill_between(fpr_arr, tpr_arr, alpha=0.08, color='#1A5276')
axes[0].set_xlabel('False Positive Rate', fontsize=11)
axes[0].set_ylabel('True Positive Rate', fontsize=11)
axes[0].set_title('ROC Curve — Anomaly Detection', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Confusion matrix heatmap
cm = confusion_matrix(y_true, y_pred)
im = axes[1].imshow(cm, cmap='Blues')
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['Pred Anomaly','Pred Normal'])
axes[1].set_yticklabels(['Actual Anomaly','Actual Normal'])
axes[1].set_title('Confusion Matrix', fontsize=12, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm[i,j]), ha='center', va='center',
                     fontsize=16, fontweight='bold',
                     color='white' if cm[i,j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.savefig('anomaly_roc_cm.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nAUC = {auc_val:.3f}")


## 5. Force Distribution Visualisation

In [ ]:
# ── Force delta distributions by event class ─────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(dW_auth,   bins=20, alpha=0.6, color='#2ECC71', label='Authorized disposal')
ax.hist(dW_unauth, bins=20, alpha=0.6, color='#E74C3C', label='Unauthorized dump')
ax.axvline(THETA_F, color='#1A5276', lw=2.5, linestyle='--',
           label=f'Threshold θ_f = {THETA_F} kg')
ax.set_xlabel('ΔW — Force delta (kg)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Force Delta Distributions by Event Class', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('anomaly_force_dist.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Summary

| Metric | Design Target | Simulated Result |
|--------|:---:|:---:|
| Precision | 89.4% | see output |
| Recall | 93.3% | see output |
| F1-score | 91.3% | see output |
| Specificity | 95.8% | see output |
| AUC | 0.961 | see output |

> **Note:** These are simulated results from the probabilistic event model.
> Physical validation requires the controlled event-injection protocol (Section 5.2).
